In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from scipy import stats as scipy_stats

# --- 比較対象ファイルの指定 ---
# 動画1: 車椅子のおばあちゃんのデータ
# 動画2: 国宝さん5のデータ
file1_path = '/Users/takahashiryoutarou/Desktop/国宝さん一覧/box/output/video2_idea5_stay_clips_integrated_analysis_20250827_085546_with_calc.csv'
file2_path = '/Users/takahashiryoutarou/Desktop/国宝さん一覧/box/output/国宝さん5_integrated_analysis_20250829_082115_with_calc.csv'

# --- データの読み込みと表示 ---
try:
    df1 = pd.read_csv(file1_path)
    df2 = pd.read_csv(file2_path)

    print("--- 動画1のデータ (車椅子おばあちゃん1) ---")
    print(f"ファイルパス: {file1_path}")
    print(f"データ形状: {df1.shape}")
    display(df1.head())
    print("--- 動画1の全カラム名 ---")
    print(df1.columns.tolist())

    print("\\n" + "--- 動画2のデータ (国宝さん5) ---")
    print(f"ファイルパス: {file2_path}")
    print(f"データ形状: {df2.shape}")
    display(df2.head())
    print("--- 動画2の全カラム名 ---")
    print(df2.columns.tolist())

except FileNotFoundError as e:
    print(f"エラー: ファイルが見つかりません。パスを確認してください。\\n{e}")
    # analyze_ideas関数が参照する前の段階でdf1, df2が存在しない場合のエラーを防ぐ
    df1, df2 = pd.DataFrame(), pd.DataFrame()



--- 動画1のデータ (車椅子おばあちゃん1) ---
ファイルパス: /Users/takahashiryoutarou/Desktop/国宝さん一覧/box/output/video2_idea5_stay_clips_integrated_analysis_20250827_085546_with_calc.csv
データ形状: (2827, 56)


,timestamp,frame_number,right_elbow_angle,right_elbow_state,left_elbow_angle,left_elbow_state,right_shoulder_angle,right_shoulder_state,left_shoulder_angle,left_shoulder_state,...,normalization_applied,prev_x,prev_y,distance_px,distance_normalized,idea2_is_spike,idea3_is_cumulative_move,idea4_is_unstable,idea5_is_composite_move,seconds
0,0.041667,1,124.667636,FLEXION,53.490165,EXTENSION,68.241271,EXTENSION,77.310500,STATIC,...,True,481.063519,451.837184,17.428229,0.175131,False,False,False,False,0
1,0.083333,2,96.976044,FLEXION,43.852175,FLEXION,81.758649,EXTENSION,78.473048,STATIC,...,True,498.410053,450.151684,15.291616,0.154372,False,False,False,False,0
2,0.125000,3,84.485807,FLEXION,58.159883,EXTENSION,83.054817,STATIC,75.127211,STATIC,...,True,483.217468,451.889176,5.919296,0.061809,False,False,False,False,0
3,0.166667,4,80.094336,STATIC,50.366088,FLEXION,79.923712,STATIC,74.356874,STATIC,...,True,488.493538,454.572673,18.482427,0.179167,False,False,False,False,0
4,0.291667,7,69.827569,FLEXION,56.826014,EXTENSION,80.065557,STATIC,74.636099,STATIC,...,True,471.093140,460.803745,43.694129,0.418148,False,False,False,False,0


\n--- 動画2のデータ (国宝さん5) ---
ファイルパス: /Users/takahashiryoutarou/Desktop/国宝さん一覧/box/output/国宝さん5_integrated_analysis_20250829_082115_with_calc.csv
データ形状: (4134, 56)


,timestamp,frame_number,right_elbow_angle,right_elbow_state,left_elbow_angle,left_elbow_state,right_shoulder_angle,right_shoulder_state,left_shoulder_angle,left_shoulder_state,...,normalization_applied,prev_x,prev_y,distance_px,distance_normalized,idea2_is_spike,idea3_is_cumulative_move,idea4_is_unstable,idea5_is_composite_move,seconds
0,0.583333,14,30.600377,EXTENSION,32.570181,EXTENSION,88.393776,STATIC,92.687898,STATIC,...,False,1016.662979,501.435885,10.325380,0.113730,False,False,False,False,0
1,0.708333,17,27.892342,STATIC,39.925510,EXTENSION,96.043720,EXTENSION,95.611913,STATIC,...,False,1022.641907,509.854074,17.546597,0.181172,False,False,False,False,0
2,0.833333,20,34.146509,EXTENSION,54.674788,EXTENSION,86.125924,FLEXION,92.001827,STATIC,...,False,1014.232559,494.453881,3.318079,0.035697,False,False,False,False,0
3,0.875000,21,27.164076,FLEXION,39.987439,FLEXION,92.418966,EXTENSION,93.598506,STATIC,...,False,1015.217285,497.622471,11.564555,0.116796,False,False,False,False,0
4,0.916667,22,21.683302,FLEXION,34.424828,FLEXION,94.731408,STATIC,95.288325,STATIC,...,False,1025.679779,502.549453,5.997256,0.059453,False,False,False,False,0


In [4]:
# --- 分析用関数の定義 ---

def analyze_ideas(csv_file: str):
    """
    指定されたCSVファイルを分析し、各種指標を計算します。
    """
    try:
        df = pd.read_csv(csv_file)
        
        # --- 事前計算 ---
        threshold = 1e-6
        df_filtered = df[(df["person_scale"].abs() > threshold)].copy()
        
        df_filtered.loc[:, "prev_x"] = df_filtered["hip_center_x"].shift(1)
        df_filtered.loc[:, "prev_y"] = df_filtered["hip_center_y"].shift(1)
        df_filtered.dropna(subset=["prev_x", "prev_y"], inplace=True)
        
        total_frames = len(df_filtered)
        if total_frames == 0:
            # データがない場合のデフォルト値を返す
            empty_stats = {"mean": np.nan, "median": np.nan, "std": np.nan, "max": np.nan, "min": np.nan}
            empty_dist_stats = {**empty_stats, "q1": np.nan, "q3": np.nan, "skew": np.nan, "kurtosis": np.nan}
            return {
                "total": 0, "idea1": 0, "idea2": 0, "idea3": 0, "idea4": 0, "idea5": 0,
                "head_shake_h": 0, "head_shake_v": 0, "knee_alert_seconds": 0, "total_seconds": 0,
                "stats": empty_dist_stats, "left_knee_stats": empty_stats, "right_knee_stats": empty_stats,
                "head_h_stats": empty_stats, "head_v_stats": empty_stats, "series": pd.Series(), "output_path": None
            }

        df_filtered.loc[:, "distance_px"] = np.sqrt(
            (df_filtered["hip_center_x"] - df_filtered["prev_x"])**2 +
            (df_filtered["hip_center_y"] - df_filtered["prev_y"])**2
        )
        df_filtered.loc[:, "distance_normalized"] = df_filtered["distance_px"] / df_filtered["person_scale"]
        
        # --- 5つのアイデアを適用 ---
        idea1_frames = (df_filtered['distance_normalized'] >= 0.1).sum()
        is_spike = df_filtered['distance_normalized'].rolling(window=30, min_periods=1).max() >= 1.5
        idea2_frames = is_spike.sum()
        is_cumulative_move = df_filtered['distance_normalized'].rolling(window=90, min_periods=1).sum() > 5.0
        idea3_frames = is_cumulative_move.sum()
        is_unstable = df_filtered['person_scale'].rolling(window=60, min_periods=1).std().fillna(0) > 50.0
        idea4_frames = is_unstable.sum()
        is_composite_move = is_spike | is_unstable
        idea5_frames = is_composite_move.sum()
        
        # --- 首振り検知の分析 ---
        head_shake_h_frames = df_filtered['head_shake_horizontal_detected'].sum()
        head_shake_v_frames = df_filtered['head_shake_vertical_detected'].sum()

        # --- 膝角度アラートの分析 ---
        KNEE_ANGLE_THRESHOLD = 90.0
        MOVING_WINDOW_SECONDS = 5
        knee_alert_seconds = 0
        total_seconds = 0
        if 'timestamp' in df_filtered.columns and not df_filtered.empty:
            total_seconds = df_filtered['timestamp'].max()
            df_filtered['seconds'] = df_filtered['timestamp'].astype(int)
            
            if not df_filtered.empty:
                knee_angles_sec = df_filtered.groupby('seconds').agg(
                    left_knee_median=('left_knee_angle', 'median'),
                    right_knee_median=('right_knee_angle', 'median')
                ).reset_index()
                knee_angles_sec['left_knee_ma5'] = knee_angles_sec['left_knee_median'].rolling(window=MOVING_WINDOW_SECONDS, min_periods=1).mean()
                knee_angles_sec['right_knee_ma5'] = knee_angles_sec['right_knee_median'].rolling(window=MOVING_WINDOW_SECONDS, min_periods=1).mean()
                is_median_low = (knee_angles_sec['left_knee_median'] < KNEE_ANGLE_THRESHOLD) | (knee_angles_sec['right_knee_median'] < KNEE_ANGLE_THRESHOLD)
                is_ma5_low = (knee_angles_sec['left_knee_ma5'] < KNEE_ANGLE_THRESHOLD) | (knee_angles_sec['right_knee_ma5'] < KNEE_ANGLE_THRESHOLD)
                knee_alert_seconds = (is_median_low | is_ma5_low).sum()

        # --- 記述統計量の計算 ---
        stats = df_filtered["distance_normalized"].describe().to_dict()
        stats.update({"skew": df_filtered["distance_normalized"].skew(), "kurtosis": df_filtered["distance_normalized"].kurtosis()})
        left_knee_stats = df_filtered["left_knee_angle"].describe().to_dict()
        right_knee_stats = df_filtered["right_knee_angle"].describe().to_dict()
        head_h_stats = df_filtered["head_horizontal_rotation_angle"].describe().to_dict()
        head_v_stats = df_filtered["head_vertical_nod_angle"].describe().to_dict()
        
        return {
            "total": total_frames, "idea1": idea1_frames, "idea2": idea2_frames, "idea3": idea3_frames,
            "idea4": idea4_frames, "idea5": idea5_frames, "head_shake_h": head_shake_h_frames, 
            "head_shake_v": head_shake_v_frames, "knee_alert_seconds": knee_alert_seconds, 
            "total_seconds": total_seconds, "stats": stats, "left_knee_stats": left_knee_stats,
            "right_knee_stats": right_knee_stats, "head_h_stats": head_h_stats,
            "head_v_stats": head_v_stats, "series": df_filtered["distance_normalized"]
        }
    except Exception as e:
        print(f"{csv_file} の処理中にエラーが発生しました: {e}", file=sys.stderr)
        return None


In [5]:
# --- 分析の実行と結果の表示 ---

if not df1.empty and not df2.empty:
    results1 = analyze_ideas(file1_path)
    results2 = analyze_ideas(file2_path)

    if results1 and results2:
        # --- アイデア比較のDataFrameを作成 ---
        total1, total2 = results1['total'], results2['total']
        def format_val(res, total):
            return f"{res} ({res/total:.1%})" if total > 0 else "0 (0.0%)"
        
        ideas_data = {
            'アイデア': ['アイデア1：単純な閾値', 'アイデア2：スパイク検知', 'アイデア3：累積移動量', 'アイデア4：安定性チェック', 'アイデア5：複合条件 (2または4)'],
            '条件': ['正規化移動量 >= 0.1', '1秒間の最大正規化移動量 >= 1.5', '3秒間の正規化移動量の合計 > 5.0', '2秒間の体幹長の標準偏差 > 50px', 'スパイク検知または安定性チェックが真'],
            '動画1 (車椅子)': [format_val(results1[f'idea{i}'], total1) for i in range(1, 6)],
            '動画2 (国宝さん5)': [format_val(results2[f'idea{i}'], total2) for i in range(1, 6)],
        }
        ideas_df = pd.DataFrame(ideas_data).set_index('アイデア')

        # --- その他の分析比較 ---
        total_seconds1, total_seconds2 = results1['total_seconds'], results2['total_seconds']
        def format_seconds(res, total):
            return f"{res} ({res/total:.1%})" if total > 0 else "0 (0.0%)"
        
        other_analysis_data = {
            '分析項目': ['水平方向の首振り (フレーム数)', '垂直方向の首振り (フレーム数)', '膝角度低下 (秒数)'],
            '動画1 (車椅子)': [
                format_val(results1['head_shake_h'], total1),
                format_val(results1['head_shake_v'], total1),
                format_seconds(results1['knee_alert_seconds'], total_seconds1),
            ],
            '動画2 (国宝さん5)': [
                format_val(results2['head_shake_h'], total2),
                format_val(results2['head_shake_v'], total2),
                format_seconds(results2['knee_alert_seconds'], total_seconds2),
            ],
        }
        other_df = pd.DataFrame(other_analysis_data).set_index('分析項目')

        # --- 統計量のヘルパー関数 ---
        def create_stats_df(stats1, stats2, columns, index_map):
            df = pd.DataFrame({columns[0]: stats1, columns[1]: stats2})
            df = df.rename(index=index_map)
            # 存在しない可能性のあるキーを安全に参照
            keys_to_drop = [k for k in ['count', '25%', '50%', '75%'] if k in df.index]
            df = df.drop(keys_to_drop)
            return df
            
        # --- 統計量の比較 ---
        stats_index_map = {'mean': '平均', 'std': '標準偏差', 'min': '最小値', 'max': '最大値', 'skew': '歪度', 'kurtosis': '尖度'}
        stats_df = create_stats_df(results1['stats'], results2['stats'], ['動画1 (車椅子)', '動画2 (国宝さん5)'], stats_index_map)
        
        # --- 膝角度の統計量比較 ---
        knee_index_map = {'mean': '平均', 'std': '標準偏差', 'min': '最小値', 'max': '最大値'}
        knee_stats_df_l = create_stats_df(results1['left_knee_stats'], results2['left_knee_stats'], ['左膝 (動画1)', '左膝 (動画2)'], knee_index_map)
        knee_stats_df_r = create_stats_df(results1['right_knee_stats'], results2['right_knee_stats'], ['右膝 (動画1)', '右膝 (動画2)'], knee_index_map)
        knee_stats_df = pd.concat([knee_stats_df_l, knee_stats_df_r], axis=1)

        # --- 首振り角度の統計量比較 ---
        head_index_map = {'mean': '平均', 'std': '標準偏差', 'min': '最小値', 'max': '最大値'}
        head_stats_df_h = create_stats_df(results1['head_h_stats'], results2['head_h_stats'], ['水平 (動画1)', '水平 (動画2)'], head_index_map)
        head_stats_df_v = create_stats_df(results1['head_v_stats'], results2['head_v_stats'], ['垂直 (動画1)', '垂直 (動画2)'], head_index_map)
        head_stats_df = pd.concat([head_stats_df_h, head_stats_df_v], axis=1)
        
        # --- 結果の表示 ---
        print("--- 動作検知アイデアの比較 ---")
        display(ideas_df)
        print("\\n" + "-"*80)
        print("--- その他の分析比較 ---")
        display(other_df)
        print("\\n" + "-"*80)
        print("--- 正規化移動量の統計量比較 ---")
        display(stats_df.style.format("{:.4f}"))
        print("\\n" + "-"*80)
        print("--- 膝角度の統計量比較 (度) ---")
        display(knee_stats_df.style.format("{:.2f}"))
        print("\\n" + "-"*80)
        print("--- 首振り角度の統計量比較 (度) ---")
        display(head_stats_df.style.format("{:.2f}"))

        # --- 仮説検定 ---
        print("\\n" + "-"*80)
        print("--- マン・ホイットニーのU検定 (移動量分布) ---")
        series1, series2 = results1['series'].dropna(), results2['series'].dropna()
        if len(series1) > 0 and len(series2) > 0:
            u_stat, p_value = scipy_stats.mannwhitneyu(series1, series2, alternative='two-sided')
            print(f"U統計量: {u_stat:.2f}, P値: {p_value:.4f}")
            print("結果: P値が0.05未満の場合、2つの動画の移動量分布には統計的に有意な差があると言えます。")
        else:
            print("データ不足のため検定を実行できませんでした。")
    else:
        print("分析の実行中にエラーが発生しました。")
else:
    print("データが読み込まれていないため、分析を実行できません。")



--- 動作検知アイデアの比較 ---


,条件,動画1 (車椅子),動画2 (国宝さん5)
アイデア,,,
アイデア1：単純な閾値,正規化移動量 >= 0.1,34 (1.2%),18 (0.4%)
アイデア2：スパイク検知,1秒間の最大正規化移動量 >= 1.5,0 (0.0%),31 (0.8%)
アイデア3：累積移動量,3秒間の正規化移動量の合計 > 5.0,61 (2.2%),40 (1.0%)
アイデア4：安定性チェック,2秒間の体幹長の標準偏差 > 50px,0 (0.0%),114 (2.8%)
アイデア5：複合条件 (2または4),スパイク検知または安定性チェックが真,0 (0.0%),114 (2.8%)


\n--------------------------------------------------------------------------------
--- その他の分析比較 ---


,動画1 (車椅子),動画2 (国宝さん5)
分析項目,,
水平方向の首振り (フレーム数),2819 (99.8%),4126 (99.8%)
垂直方向の首振り (フレーム数),2819 (99.8%),4039 (97.7%)
膝角度低下 (秒数),94 (77.0%),136 (78.3%)


\n--------------------------------------------------------------------------------
--- 正規化移動量の統計量比較 ---


...

\n--------------------------------------------------------------------------------
--- 膝角度の統計量比較 (度) ---


,左膝 (動画1),左膝 (動画2),右膝 (動画1),右膝 (動画2)
平均,85.21,158.09,120.31,75.84
標準偏差,20.66,13.73,17.20,35.57
最小値,42.01,80.99,52.98,27.83
最大値,157.29,178.46,153.20,173.72


\n--------------------------------------------------------------------------------
--- 首振り角度の統計量比較 (度) ---


,水平 (動画1),水平 (動画2),垂直 (動画1),垂直 (動画2)
平均,-50.97,60.60,-38.06,-29.10
標準偏差,15.70,121.11,8.97,14.71
最小値,-175.37,-179.97,-66.83,-163.03
最大値,174.37,179.98,12.27,46.60


\n--------------------------------------------------------------------------------
--- マン・ホイットニーのU検定 (移動量分布) ---
U統計量: 7476846.00, P値: 0.0000
結果: P値が0.05未満の場合、2つの動画の移動量分布には統計的に有意な差があると言えます。


In [6]:
# --- 生データの一部を表示 ---

if not df1.empty and not df2.empty:
    columns_to_show = [
        'timestamp',
        'left_knee_angle',
        'right_knee_angle',
        'head_horizontal_rotation_angle',
        'head_vertical_nod_angle'
    ]

    print("--- 動画1の生データ (車椅子おばあちゃん1, 最初の20行) ---")
    if all(col in df1.columns for col in columns_to_show):
        display(df1[columns_to_show].head(20))
    else:
        print("指定されたカラムの一部が動画1のデータフレームに存在しません。")

    print("\\n" + "--- 動画2の生データ (国宝さん5, 最初の20行) ---")
    if all(col in df2.columns for col in columns_to_show):
        display(df2[columns_to_show].head(20))
    else:
        print("指定されたカラムの一部が動画2のデータフレームに存在しません。")
else:
    print("データが読み込まれていないため、生データを表示できません。")



--- 動画1の生データ (車椅子おばあちゃん1, 最初の20行) ---


,timestamp,left_knee_angle,right_knee_angle,head_horizontal_rotation_angle,head_vertical_nod_angle
0,0.041667,108.203293,99.602284,-152.660504,0.854597
1,0.083333,86.943964,88.977393,-51.727687,-1.326727
2,0.125000,125.574753,101.233761,31.233910,2.840559
3,0.166667,124.085546,94.260629,4.299564,4.435116
4,0.291667,112.220433,85.406963,70.722622,10.642737
5,0.333333,115.338794,77.173711,57.705628,12.269815
6,0.708333,98.655525,73.021882,-36.337181,-54.616294
7,0.750000,117.208028,94.294090,-37.994518,-49.132351
8,0.791667,117.695559,100.997152,-40.091065,-49.850822
9,0.958333,118.120664,108.086246,-60.565076,-47.209119


\n--- 動画2の生データ (国宝さん5, 最初の20行) ---


,timestamp,left_knee_angle,right_knee_angle,head_horizontal_rotation_angle,head_vertical_nod_angle
0,0.583333,130.562788,73.367408,-51.432309,-104.476259
1,0.708333,123.082210,59.437883,-112.224785,-79.441310
2,0.833333,137.305876,64.556048,37.520023,-94.890047
3,0.875000,134.126333,61.378735,25.173679,-80.896682
4,0.916667,135.045441,74.827436,32.292110,-89.215274
5,0.958333,131.082966,84.511801,33.667685,-84.911167
6,1.000000,130.522977,81.460311,36.474249,-84.311348
7,1.041667,135.813741,82.047237,32.361918,-80.700812
8,1.083333,138.482664,87.096554,29.800952,-77.416502
9,1.125000,138.859743,89.934566,40.437804,-91.298923


In [7]:
# --- データフレーム全体の基本統計量 ---

if not df1.empty and not df2.empty:
    print("--- 動画1の基本統計量 (車椅子おばあちゃん1) ---")
    display(df1.describe())

    print("\\n" + "--- 動画2の基本統計量 (国宝さん5) ---")
    display(df2.describe())
else:
    print("データが読み込まれていないため、基本統計量を計算できません。")



--- 動画1の基本統計量 (車椅子おばあちゃん1) ---


,timestamp,frame_number,right_elbow_angle,left_elbow_angle,right_shoulder_angle,left_shoulder_angle,right_hip_angle,left_hip_angle,right_knee_angle,left_knee_angle,...,stay_duration,hip_confidence,torso_length,shoulder_width,person_scale,prev_x,prev_y,distance_px,distance_normalized,seconds
count,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000,...,2827.000000,2827.000000,2827.000000,2827.0,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000,2827.000000
mean,62.707935,1504.990449,49.014946,119.000352,84.729303,55.063869,85.961111,82.486984,120.306528,85.213498,...,62.707935,0.998525,92.397887,0.0,92.397887,572.051584,328.431265,1.416867,0.015043,62.229926
std,34.717991,833.231774,22.146544,21.949525,3.841949,12.813336,8.734826,10.511600,17.205493,20.657495,...,34.717991,0.000881,6.509616,0.0,6.509616,26.579720,25.977976,3.447578,0.032982,34.716385
min,0.041667,1.000000,3.339441,43.852175,46.430189,20.204142,43.508518,44.256057,52.982185,42.010358,...,0.041667,0.995336,74.666055,0.0,74.666055,471.006603,304.520513,0.002935,0.000030,0.000000
25%,33.604167,806.500000,31.950213,106.951683,84.254370,46.524459,82.430391,78.763451,113.687539,69.518223,...,33.604167,0.998002,86.973088,0.0,86.973088,547.394848,316.640723,0.316580,0.003448,33.000000
50%,63.125000,1515.000000,48.276794,127.578511,85.273486,53.109883,86.945616,83.942380,124.014741,81.009032,...,63.125000,0.998550,92.731587,0.0,92.731587,568.671417,323.865527,0.709021,0.007757,63.000000
75%,92.645833,2223.500000,62.861922,133.441042,86.422404,61.494676,90.634473,88.319857,132.161007,98.312585,...,92.645833,0.999176,96.648901,0.0,96.648901,587.903986,332.469732,1.571416,0.016762,92.000000
max,122.083333,2930.000000,128.090891,160.400277,96.874007,106.580659,128.500434,129.319103,153.195476,157.289881,...,122.083333,0.999929,127.606476,0.0,127.606476,642.873421,645.076997,133.757751,1.238669,122.000000


\n--- 動画2の基本統計量 (国宝さん5) ---


,timestamp,frame_number,right_elbow_angle,left_elbow_angle,right_shoulder_angle,left_shoulder_angle,right_hip_angle,left_hip_angle,right_knee_angle,left_knee_angle,...,stay_duration,hip_confidence,torso_length,shoulder_width,person_scale,prev_x,prev_y,distance_px,distance_normalized,seconds
count,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000,...,4134.000000,4134.000000,4134.000000,4134.0,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000,4134.000000
mean,86.977106,2086.580552,52.987744,128.862056,84.274458,44.768618,111.039093,159.374688,75.841084,158.086003,...,81.512573,0.992552,113.788222,0.0,113.788222,682.750369,472.897835,1.417430,0.012426,86.479923
std,49.758642,1193.709683,42.958558,30.505599,8.365533,12.181237,13.751958,12.128770,35.561914,13.738645,...,49.604714,0.053783,24.709941,0.0,24.709941,51.510755,40.181708,11.649775,0.208292,49.756012
min,0.583333,14.000000,4.761975,26.530782,6.740555,5.726642,90.540281,83.130783,27.829429,80.985467,...,0.000000,0.229467,41.242585,0.0,41.242585,586.219177,436.240590,0.001231,0.000011,0.000000
25%,43.903750,1053.250000,18.736768,105.169598,84.386296,39.446040,100.858238,152.790106,47.697332,152.173618,...,38.360417,0.998887,102.815705,0.0,102.815705,660.186319,456.785104,0.164893,0.001518,43.000000
50%,86.974167,2086.500000,37.336517,139.340945,85.030840,44.605389,107.308612,162.085542,67.529120,161.618427,...,81.430833,0.999312,112.088836,0.0,112.088836,683.140717,469.429235,0.455875,0.004072,86.000000
75%,130.044583,3119.750000,79.715182,154.108266,86.336046,49.994283,118.017787,168.411947,99.234249,168.410299,...,124.499583,0.999519,118.628424,0.0,118.628424,709.252357,478.430026,1.134376,0.010114,130.000000
max,173.655000,4166.000000,176.991705,171.026065,97.236158,96.566465,176.691620,178.626644,173.722468,178.463620,...,166.611667,0.999898,404.518762,0.0,404.518762,1178.388863,985.856137,529.351899,12.835081,173.000000
